# Explore Event Precipitation and Meteorology

In [1]:
import os
os.chdir('/home/dlhogan/projects/phd-repos/S3-precipitation-rodeo/')

# general
import glob
import datetime as dt
from pathlib import Path

# data 
import xarray as xr 
import numpy as np
import pandas as pd

# plotting
import matplotlib.pyplot as plt
import plotly.express as px 
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

# Configure Plotly for Jupyter notebooks
pio.renderers.default = "notebook"
# Alternative renderers you can try if "notebook" doesn't work:
# pio.renderers.default = "plotly_mimetype+notebook"
# pio.renderers.default = "jupyter_lab"

# helper tools
from metpy import calc, units
import scipy.stats as stats
from sklearn.linear_model import LinearRegression

In [5]:
def get_file_destination(DATA_DIR, SRC, PRODUCT, WITH_MET, RAW_OR_NORMALIZED):
    if WITH_MET and RAW_OR_NORMALIZED == "raw":
        WITH_MET = "_with_raw_met"
    elif WITH_MET and RAW_OR_NORMALIZED == "normalized":
        WITH_MET = "_with_normalized_met"
    else:
        WITH_MET = ""

    if PRODUCT == "gridded":
        PRODUCT_NAME = "_gridded"
        FOLDER_NAME = "gridded_events"
    elif PRODUCT == "events":
        PRODUCT_NAME = ""
        FOLDER_NAME = "events"
    else:
        PRODUCT_NAME = ""
    if SRC in ['asfs', 'sos']:
        SITE = 'kettle_ponds'
    elif SRC in ['bb', 'sail']:
        SITE = 'gothic'
    else:
        SITE = input("Enter site name (gothic or kettle_ponds): ")
    return DATA_DIR / SITE / f"{FOLDER_NAME}{WITH_MET}" / f"{SITE}{PRODUCT_NAME}_precipitation_event_comparisons_{SRC}{WITH_MET}.nc"

In [7]:
DATA_DIR = '/storage/dlhogan/precipitation-rodeo/data/for_analysis'
# Define your root and sites
DATA_DIR = Path(DATA_DIR)
SRC = "sail" # one of [bb, sail, asfs, sos, '']
PRODUCT = "events" # or events
WITH_MET = True  # or True
RAW_OR_NORMALIZED = "raw"  # or raw
file_dest = get_file_destination(DATA_DIR, SRC, PRODUCT, WITH_MET, RAW_OR_NORMALIZED)
print(f"Loading data from: {file_dest}")
ds = xr.open_dataset(file_dest)

Loading data from: /storage/dlhogan/precipitation-rodeo/data/for_analysis/gothic/events_with_raw_met/gothic_precipitation_event_comparisons_sail_with_raw_met.nc


In [9]:
gothic_ppt_ds = xr.open_dataset('/storage/dlhogan/precipitation-rodeo/data/processed/final/gothic_precipitation_30min.nc')
gothic_met_ds = xr.open_dataset('/storage/dlhogan/precipitation-rodeo/data/processed/SAIL/met_30min.nc')

In [25]:
# set benchmark site 
BENCHMARK = 'billy_barr_precip'
EVENTS = slice(1,11) # top 10 events
INSTRUMENT = "sail_pluvio"

In [37]:
event_ds = ds.sel(event_id=1, test_instrument=INSTRUMENT, benchmark=BENCHMARK)
event_ds['start_time'].values

np.datetime64('2021-10-07T14:30:00.000000000')

In [45]:
start, end = pd.to_datetime(event_ds['start_time'].values), pd.to_datetime(event_ds['end_time'].values)
event_ppt = gothic_ppt_ds.sel(time=slice(start, end))
event_met = gothic_met_ds.sel(time=slice(start, end))

In [50]:
# make a plotly plot of the event with ppt and met data
fig = make_subplots(rows=3, cols=1, shared_xaxes=True, subplot_titles=("Precipitation", "Air Temperature", "Wind Speed"))
fig.add_trace(go.Scatter(x=event_ppt['time'].values, y=event_ppt['billy_barr_precip'].cumsum().values, name='BB (mm)'), row=1, col=1)
fig.add_trace(go.Scatter(x=event_ppt['time'].values, y=event_ppt['sail_pluvio'].cumsum().values, name='Tipping Bucket (mm)'), row=1, col=1)
fig.add_trace(go.Scatter(x=event_met['time'].values, y=event_met['temp_mean'].values, mode='lines+markers', name='Air Temperature (°C)'), row=2, col=1)
fig.add_trace(go.Scatter(x=event_met['time'].values, y=event_met['wspd_vec_mean'].values, mode='lines+markers', name='Wind Speed (m/s)'), row=3, col=1)
fig.update_layout(height=800, width=800, title_text=f'Event ID: {event_ds["event_id"].values} from {event_ds["start_time"].values} to {event_ds["end_time"].values}')
fig.show()